In [11]:
import pandas as pd
import numpy as np

In [12]:
df = pd.read_excel('Online Retail.xlsx')

df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [13]:
# Number of rows and columns
print("Dataset shape:", df.shape)

Dataset shape: (541909, 8)


In [14]:
# Data types and non-null counts
print("\nDataset information:")
df.info()


Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[ns]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB


In [15]:
# Check missing values
print("Missing values by column:")
print(df.isnull().sum())

Missing values by column:
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64


In [16]:
# Check duplicate records
print("\nDuplicate rows:", df.duplicated().sum())


Duplicate rows: 5268


In [17]:
# Check numerical summaries
print("\nNumerical summary:")
display(df[["Quantity", "UnitPrice"]].describe())


Numerical summary:


,Quantity,UnitPrice
count,541909.000000,541909.000000
mean,9.552250,4.611114
std,218.081158,96.759853
min,-80995.000000,-11062.060000
25%,1.000000,1.250000
50%,3.000000,2.080000
75%,10.000000,4.130000
max,80995.000000,38970.000000


In [18]:
# Check date range
print("\nTransaction date range:")
print("Earliest:", df["InvoiceDate"].min())
print("Latest:", df["InvoiceDate"].max())


Transaction date range:
Earliest: 2010-12-01 08:26:00
Latest: 2011-12-09 12:50:00


In [19]:
# Count unique values
print("\nUnique values:")
print("Invoices:", df["InvoiceNo"].nunique())
print("Products:", df["StockCode"].nunique())
print("Customers:", df["CustomerID"].nunique())
print("Countries:", df["Country"].nunique())


Unique values:
Invoices: 25900
Products: 4070
Customers: 4372
Countries: 38


In [20]:
# Identify cancellation invoices
df["InvoiceNo"] = df["InvoiceNo"].astype(str)

is_cancellation = df["InvoiceNo"].str.startswith("C")

print("Cancellation invoice rows:", is_cancellation.sum())
print("Non-cancellation invoice rows:", (~is_cancellation).sum())

# Inspect negative and zero quantities
print("\nNegative quantity rows:", (df["Quantity"] < 0).sum())
print("Zero quantity rows:", (df["Quantity"] == 0).sum())

# Inspect negative and zero unit prices
print("\nNegative unit price rows:", (df["UnitPrice"] < 0).sum())
print("Zero unit price rows:", (df["UnitPrice"] == 0).sum())

Cancellation invoice rows: 9288
Non-cancellation invoice rows: 532621

Negative quantity rows: 10624
Zero quantity rows: 0

Negative unit price rows: 2
Zero unit price rows: 2515


In [21]:
# Create a separate working copy and remove exact duplicate rows
clean_df = df.drop_duplicates().copy()

In [22]:
# Standardize invoice numbers and customer IDs
clean_df["InvoiceNo"] = clean_df["InvoiceNo"].astype(str).str.strip()
clean_df["CustomerID"] = pd.to_numeric(
    clean_df["CustomerID"], errors="coerce"
).astype("Int64")

In [23]:
# Add transaction-level fields
clean_df["IsCancellation"] = clean_df["InvoiceNo"].str.startswith("C")
clean_df["Revenue"] = clean_df["Quantity"] * clean_df["UnitPrice"]

In [24]:
# Qualifying sales lines for our main analysis
sales_df = clean_df.loc[
    (~clean_df["IsCancellation"]) &
    (clean_df["Quantity"] > 0) &
    (clean_df["UnitPrice"] > 0)
].copy()

In [25]:
# Customer-identifiable sales for RFM
customer_sales_df = sales_df.dropna(subset=["CustomerID"]).copy()

In [26]:
print("Original rows:", len(df))
print("Rows after deduplication:", len(clean_df))
print("Qualifying sales rows:", len(sales_df))
print("Qualifying sales rows with customer IDs:", len(customer_sales_df))
print("Customers available for RFM:",
      customer_sales_df["CustomerID"].nunique())

Original rows: 541909
Rows after deduplication: 536641
Qualifying sales rows: 524878
Qualifying sales rows with customer IDs: 392692
Customers available for RFM: 4338


In [27]:
# Reference date: one day after the latest transaction in the dataset
reference_date = clean_df["InvoiceDate"].max().normalize() + pd.Timedelta(days=1)

customer_rfm = customer_sales_df.groupby("CustomerID").agg(
    LastPurchaseDate=("InvoiceDate", "max"),
    Frequency=("InvoiceNo", "nunique"),
    Monetary=("Revenue", "sum")
).reset_index()

customer_rfm["Recency"] = (
    reference_date - customer_rfm["LastPurchaseDate"].dt.normalize()
).dt.days

# Put columns in a convenient order
customer_rfm = customer_rfm[
    ["CustomerID", "LastPurchaseDate", "Recency", "Frequency", "Monetary"]
]

print("Customers in RFM:", len(customer_rfm))
display(customer_rfm.head())

Customers in RFM: 4338


,CustomerID,LastPurchaseDate,Recency,Frequency,Monetary
0,12346,2011-01-18 10:01:00,326,1,77183.60
1,12347,2011-12-07 15:52:00,3,7,4310.00
2,12348,2011-09-25 13:13:00,76,4,1797.24
3,12349,2011-11-21 09:51:00,19,1,1757.55
4,12350,2011-02-02 16:01:00,311,1,334.40


In [28]:
print("\nRFM summary:")
display(customer_rfm[["Recency", "Frequency", "Monetary"]].describe())


RFM summary:


,Recency,Frequency,Monetary
count,4338.000000,4338.000000,4338.000000
mean,93.059474,4.272015,2048.688081
std,100.012264,7.697998,8985.230220
min,1.000000,1.000000,3.750000
25%,18.000000,1.000000,306.482500
50%,51.000000,2.000000,668.570000
75%,142.750000,5.000000,1660.597500
max,374.000000,209.000000,280206.020000


In [29]:
# Rank first so tied values can still be divided into five groups
customer_rfm["R_Score"] = pd.qcut(
    customer_rfm["Recency"].rank(method="first", ascending=True),
    5, labels=[5, 4, 3, 2, 1]
).astype(int)

customer_rfm["F_Score"] = pd.qcut(
    customer_rfm["Frequency"].rank(method="first"),
    5, labels=[1, 2, 3, 4, 5]
).astype(int)

customer_rfm["M_Score"] = pd.qcut(
    customer_rfm["Monetary"].rank(method="first"),
    5, labels=[1, 2, 3, 4, 5]
).astype(int)

# Combined RFM score
customer_rfm["RFM_Score"] = (
    customer_rfm["R_Score"].astype(str) +
    customer_rfm["F_Score"].astype(str) +
    customer_rfm["M_Score"].astype(str)
)

In [30]:
# Business-defined segments; first matching rule takes priority
conditions = [
    (customer_rfm["R_Score"] >= 4) &
    (customer_rfm["F_Score"] >= 4) &
    (customer_rfm["M_Score"] >= 4),

    (customer_rfm["R_Score"] >= 3) &
    (customer_rfm["F_Score"] >= 4),

    (customer_rfm["R_Score"] >= 3) &
    (customer_rfm["M_Score"] == 5) &
    (customer_rfm["F_Score"] <= 3),

    (customer_rfm["R_Score"] >= 3) &
    (customer_rfm["F_Score"].between(2, 3)),

    (customer_rfm["R_Score"] == 5) &
    (customer_rfm["F_Score"] == 1),

    (customer_rfm["R_Score"] <= 2) &
    (customer_rfm["F_Score"] >= 3),

    (customer_rfm["R_Score"] <= 2) &
    (customer_rfm["F_Score"] <= 2),

    (customer_rfm["R_Score"] == 4) &
    (customer_rfm["F_Score"] == 1)
]

segment_names = [
    "Champions",
    "Loyal Customers",
    "Big Spenders",
    "Potential Loyalists",
    "New Customers",
    "At Risk",
    "Hibernating",
    "Promising"
]

customer_rfm["Segment"] = np.select(
    conditions, segment_names, default="Need Attention"
)

print("Customer segment counts:")
display(
    customer_rfm["Segment"]
    .value_counts()
    .rename_axis("Segment")
    .reset_index(name="Customers")
)

Customer segment counts:


,Segment,Customers
0,Hibernating,1072
1,Champions,941
2,Potential Loyalists,819
3,At Risk,663
4,Loyal Customers,504
5,Need Attention,164
6,Promising,94
7,New Customers,41
8,Big Spenders,40


In [31]:
segment_summary = customer_rfm.groupby("Segment", as_index=False).agg(
    Customers=("CustomerID", "nunique"),
    GrossSales=("Monetary", "sum"),
    AverageSalesPerCustomer=("Monetary", "mean"),
    AverageRecencyDays=("Recency", "mean"),
    AverageInvoices=("Frequency", "mean")
)

segment_summary["CustomerSharePct"] = (
    segment_summary["Customers"] / customer_rfm["CustomerID"].nunique() * 100
)

segment_summary["SalesSharePct"] = (
    segment_summary["GrossSales"] / customer_rfm["Monetary"].sum() * 100
)

print("Validation checks")
print("RFM rows:", len(customer_rfm))
print("Unique customer IDs:", customer_rfm["CustomerID"].nunique())
print("Missing RFM values:", customer_rfm[
    ["Recency", "Frequency", "Monetary", "R_Score", "F_Score", "M_Score", "Segment"]
].isna().sum().sum())
print("Non-positive frequency:", (customer_rfm["Frequency"] <= 0).sum())
print("Non-positive monetary value:", (customer_rfm["Monetary"] <= 0).sum())

Validation checks
RFM rows: 4338
Unique customer IDs: 4338
Missing RFM values: 0
Non-positive frequency: 0
Non-positive monetary value: 0


In [32]:
print("\nSegment summary:")
display(segment_summary.sort_values("GrossSales", ascending=False))

print("\nTotal RFM customers:", segment_summary["Customers"].sum())
print("Total RFM gross sales:", round(segment_summary["GrossSales"].sum(), 2))


Segment summary:


,Segment,Customers,GrossSales,AverageSalesPerCustomer,AverageRecencyDays,AverageInvoices,CustomerSharePct,SalesSharePct
2,Champions,941,5741913.580,6101.927290,12.981934,11.214665,21.692024,64.608739
4,Loyal Customers,504,918045.601,1821.519050,37.365079,5.119048,11.618257,10.329965
0,At Risk,663,827416.571,1247.988795,150.944193,3.417798,15.283541,9.310196
3,Hibernating,1072,523208.532,488.067660,217.469216,1.101679,24.711849,5.887209
7,Potential Loyalists,819,474267.480,579.081172,32.620269,1.902320,18.879668,5.336518
1,Big Spenders,40,299094.750,7477.368750,35.300000,2.375000,0.922084,3.365452
5,Need Attention,164,63499.720,387.193415,53.621951,1.000000,3.780544,0.714507
8,Promising,94,27452.720,292.050213,23.510638,1.000000,2.166897,0.308901
6,New Customers,41,12309.940,300.242439,7.536585,1.000000,0.945136,0.138513



Total RFM customers: 4338
Total RFM gross sales: 8887208.89


In [33]:
# Create month field for trend analysis
sales_df["Month"] = sales_df["InvoiceDate"].dt.to_period("M").astype(str)

monthly_sales = sales_df.groupby("Month", as_index=False).agg(
    GrossSales=("Revenue", "sum"),
    Invoices=("InvoiceNo", "nunique"),
    Customers=("CustomerID", "nunique")
)

In [34]:
country_sales = sales_df.groupby("Country", as_index=False).agg(
    GrossSales=("Revenue", "sum"),
    Invoices=("InvoiceNo", "nunique"),
    Customers=("CustomerID", "nunique")
).sort_values("GrossSales", ascending=False)

In [38]:
# Export the datasets needed for Power BI
exports = {
    "SalesTransactions.csv": sales_df,
    "CustomerRFM.csv": customer_rfm,
    "SegmentSummary.csv": segment_summary,
    "MonthlySales.csv": monthly_sales,
    "CountrySales.csv": country_sales
}

for filename, data in exports.items():
    data.to_csv(filename, index=False)
    print(f"Exported: {filename} ({len(data):,} rows)")

Exported: SalesTransactions.csv (524,878 rows)
Exported: CustomerRFM.csv (4,338 rows)
Exported: SegmentSummary.csv (9 rows)
Exported: MonthlySales.csv (13 rows)
Exported: CountrySales.csv (38 rows)
